# Erlang C Dataset Tests

Place this notebook inside the `tests` folder. Place the four CSV files inside `tests/data`. Run the setup cell first, and then run Tests 1–10 one at a time. Each cell prints PASS or FAIL.

In [53]:
from pathlib import Path
import pandas as pd

DATA_CANDIDATES = [Path.cwd() / "data", Path.cwd() / "tests" / "data"]
DATA_DIR = next((folder for folder in DATA_CANDIDATES if folder.exists()), DATA_CANDIDATES[0])
CDR_FILES = {
    2021: DATA_DIR / "cdr_2021.csv",
    2022: DATA_DIR / "cdr_2022.csv",
    2023: DATA_DIR / "cdr_2023.csv",
    2024: DATA_DIR / "cdr_2024.csv",
}

def read_cdr(path):
    return pd.read_csv(path, header=None, dtype=str)

def show_result(passed, success, failure):
    print("PASS:", success) if passed else print("FAIL:", failure)
    return passed

print("Setup complete")
print("Expected dataset folder:", DATA_DIR.resolve())

Setup complete
Expected dataset folder: C:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\tests\data


## Test 1 — All four files exist

In [54]:
missing = [p.name for p in CDR_FILES.values() if not p.exists()]
for year, path in CDR_FILES.items():
    print(year, path.name, "FOUND" if path.exists() else "MISSING")
show_result(not missing, "All four files exist.", f"Missing files: {missing}")

2021 cdr_2021.csv FOUND
2022 cdr_2022.csv FOUND
2023 cdr_2023.csv FOUND
2024 cdr_2024.csv FOUND
PASS: All four files exist.


True

## Test 2 — Files are not empty

In [55]:
empty = []
for path in CDR_FILES.values():
    if not path.exists():
        empty.append(f"{path.name} (missing)")
        continue
    size = path.stat().st_size
    print(f"{path.name}: {size:,} bytes")
    if size == 0: empty.append(path.name)
show_result(not empty, "No file is empty.", f"Empty or missing: {empty}")

cdr_2021.csv: 12,169,352 bytes
cdr_2022.csv: 16,602,577 bytes
cdr_2023.csv: 17,041,692 bytes
cdr_2024.csv: 19,329,939 bytes
PASS: No file is empty.


True

## Test 3 — Files are under the 25 MB upload limit

In [56]:
limit = 25 * 1024 * 1024
too_large = []
for path in CDR_FILES.values():
    if not path.exists():
        too_large.append(f"{path.name} (missing)")
        continue
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{path.name}: {size_mb:.2f} MB")
    if path.stat().st_size > limit: too_large.append(path.name)
show_result(not too_large, "All files are within 25 MB.", f"Over limit or missing: {too_large}")

cdr_2021.csv: 11.61 MB
cdr_2022.csv: 15.83 MB
cdr_2023.csv: 16.25 MB
cdr_2024.csv: 18.43 MB
PASS: All files are within 25 MB.


True

## Test 4 — CSV files can be read

In [57]:
read_errors = {}
for path in CDR_FILES.values():
    try:
        frame = read_cdr(path)
        print(f"{path.name}: read {len(frame):,} rows")
    except Exception as error:
        read_errors[path.name] = str(error)
        print(f"{path.name}: ERROR — {error}")
show_result(not read_errors, "All CSV files can be read.", f"Read errors: {read_errors}")

cdr_2021.csv: read 151,825 rows
cdr_2022.csv: read 206,623 rows
cdr_2023.csv: read 212,092 rows
cdr_2024.csv: read 240,375 rows
PASS: All CSV files can be read.


True

## Test 5 — Each dataset has seven columns

In [58]:
wrong_columns = {}
for path in CDR_FILES.values():
    try:
        count = read_cdr(path).shape[1]
        print(f"{path.name}: {count} columns")
        if count != 7: wrong_columns[path.name] = count
    except Exception as error:
        wrong_columns[path.name] = str(error)
show_result(not wrong_columns, "Every dataset has seven columns.", f"Problems: {wrong_columns}")

cdr_2021.csv: 7 columns
cdr_2022.csv: 7 columns
cdr_2023.csv: 7 columns
cdr_2024.csv: 7 columns
PASS: Every dataset has seven columns.


True

## Test 6 — Date values are valid

In [59]:
date_quality_problems = {}

minimum_valid_percentage = 99.0

column_names = [
    "source",
    "destination",
    "call_datetime",
    "duration",
    "disposition",
    "unique_id",
    "caller_id",
]

for path in CDR_FILES.values():
    try:
        frame = read_cdr(path)
        frame.columns = column_names

        parsed_dates = pd.to_datetime(
            frame["call_datetime"],
            format="%Y-%b-%d %I:%M:%S %p",
            errors="coerce",
        )

        valid_mask = parsed_dates.notna()
        invalid_mask = parsed_dates.isna()

        valid_count = int(valid_mask.sum())
        invalid_count = int(invalid_mask.sum())
        total_count = len(frame)

        valid_percentage = (
            valid_count / total_count
        ) * 100

        # Simulate the application's cleaning behaviour
        cleaned_frame = frame.loc[
            valid_mask
        ].copy()

        print("\n" + "=" * 70)
        print(f"File: {path.name}")
        print(f"Total rows: {total_count:,}")
        print(f"Valid dates: {valid_count:,}")
        print(f"Invalid dates removed: {invalid_count:,}")
        print(
            f"Valid percentage: "
            f"{valid_percentage:.3f}%"
        )
        print(
            f"Rows remaining after cleaning: "
            f"{len(cleaned_frame):,}"
        )

        print("\nOne correct row:")

        valid_example = frame.loc[
            valid_mask
        ].head(1).copy()

        valid_example.insert(
            0,
            "csv_row_number",
            valid_example.index + 1,
        )

        display(valid_example)

        if invalid_count > 0:
            print("\nThree incorrect rows:")

            invalid_examples = frame.loc[
                invalid_mask
            ].head(3).copy()

            invalid_examples.insert(
                0,
                "csv_row_number",
                invalid_examples.index + 1,
            )

            display(invalid_examples)

        passed = (
            valid_percentage
            >= minimum_valid_percentage
            and len(cleaned_frame) == valid_count
            and not cleaned_frame.empty
        )

        if not passed:
            date_quality_problems[path.name] = {
                "valid_percentage": round(
                    valid_percentage,
                    3,
                ),
                "invalid_dates": invalid_count,
            }

    except Exception as error:
        date_quality_problems[path.name] = str(error)

show_result(
    not date_quality_problems,
    (
        "All datasets contain at least "
        f"{minimum_valid_percentage}% valid dates, "
        "and invalid dates can be removed safely."
    ),
    f"Date-quality problems: {date_quality_problems}",
)


File: cdr_2021.csv
Total rows: 151,825
Valid dates: 151,657
Invalid dates removed: 168
Valid percentage: 99.889%
Rows remaining after cleaning: 151,657

One correct row:


,csv_row_number,source,destination,call_datetime,duration,disposition,unique_id,caller_id
0,1,1001,2007,2021-Jan-01 01:05:22 AM,00:04:01,Answered,1609443322.1001,0742096946



Three incorrect rows:


,csv_row_number,source,destination,call_datetime,duration,disposition,unique_id,caller_id
134,135,1001,2004,NaN,00:01:32,Abandoned,1609527307.1135,0787640332
562,563,1006,2020,0000-00-00 00:00:00,00:08:48,Answered,1609727614.1563,0720369209
1412,1413,1003,2007,NaN,00:02:02,Answered,1609825914.2413,0753417432



File: cdr_2022.csv
Total rows: 206,623
Valid dates: 206,413
Invalid dates removed: 210
Valid percentage: 99.898%
Rows remaining after cleaning: 206,413

One correct row:


...

,csv_row_number,source,destination,call_datetime,duration,disposition,unique_id,caller_id
0,1,1004,2024,2024-Jan-01 12:28:48 AM,00:02:06,Answered,1704049128.1001,0756562582



Three incorrect rows:


,csv_row_number,source,destination,call_datetime,duration,disposition,unique_id,caller_id
1840,1841,1002,2037,0000-00-00 00:00:00,00:04:28,Answered,1704270653.2841,0760819999
2251,2252,1001,2036,NaN,00:01:27,Answered,1704303628.3252,0513580508
2367,2368,1002,2048,2024-01-04 25:61:00,00:05:43,Answered,1704339170.3368,0775999545


PASS: All datasets contain at least 99.0% valid dates, and invalid dates can be removed safely.


True

## Test 7 — Every file contains its expected year

In [60]:
year_errors = {}

for expected_year, path in CDR_FILES.items():
    try:
        frame = read_cdr(path)

        parsed_dates = pd.to_datetime(
            frame.iloc[:, 2],
            format="%Y-%b-%d %I:%M:%S %p",
            errors="coerce",
        )

        years_found = sorted(
            parsed_dates
            .dropna()
            .dt.year
            .unique()
            .tolist()
        )

        print("\n" + "=" * 60)
        print(f"File: {path.name}")
        print(f"Expected year: {expected_year}")
        print(f"Years found: {years_found}")

        passed = years_found == [expected_year]

        if passed:
            print("PASS: Dataset contains only the expected year.")
        else:
            print("FAIL: Dataset contains an unexpected year.")
            year_errors[path.name] = {
                "expected": expected_year,
                "found": years_found,
            }

    except Exception as error:
        year_errors[path.name] = str(error)
        print(f"FAIL: {path.name} could not be tested: {error}")

show_result(
    not year_errors,
    "Every dataset contains its expected year.",
    f"Year problems: {year_errors}",
)


File: cdr_2021.csv
Expected year: 2021
Years found: [2021]
PASS: Dataset contains only the expected year.

File: cdr_2022.csv
Expected year: 2022
Years found: [2022]
PASS: Dataset contains only the expected year.

File: cdr_2023.csv
Expected year: 2023
Years found: [2023]
PASS: Dataset contains only the expected year.

File: cdr_2024.csv
Expected year: 2024
Years found: [2024]
PASS: Dataset contains only the expected year.
PASS: Every dataset contains its expected year.


True

## Test 8 — Disposition values are recognized

In [61]:
allowed = {"answered", "abandoned", "no answer", "busy", "voicemail", "failed"}
unexpected = {}
for path in CDR_FILES.values():
    try:
        frame = read_cdr(path)
        values = set(frame.iloc[:, 4].dropna().str.strip().str.casefold().unique())
        extra = sorted(values - allowed)
        print(f"{path.name}: {sorted(values)}")
        if extra: unexpected[path.name] = extra
    except Exception as error:
        unexpected[path.name] = str(error)
show_result(not unexpected, "All disposition values are recognized.", f"Unexpected values: {unexpected}")

cdr_2021.csv: ['abandoned', 'answered', 'busy', 'failed', 'no answer', 'voicemail']
cdr_2022.csv: ['abandoned', 'answered', 'busy', 'failed', 'no answer', 'voicemail']
cdr_2023.csv: ['abandoned', 'answered', 'busy', 'failed', 'no answer', 'voicemail']
cdr_2024.csv: ['abandoned', 'answered', 'busy', 'failed', 'no answer', 'voicemail']
PASS: All disposition values are recognized.


True

## Test 9 — Unique IDs are not duplicated

In [62]:
duplicates = {}
for path in CDR_FILES.values():
    try:
        frame = read_cdr(path)
        count = int(frame.iloc[:, 5].duplicated().sum())
        print(f"{path.name}: {count:,} duplicated unique IDs")
        if count: duplicates[path.name] = count
    except Exception as error:
        duplicates[path.name] = str(error)
show_result(not duplicates, "No duplicated unique IDs were found.", f"Duplicate counts: {duplicates}")

cdr_2021.csv: 0 duplicated unique IDs
cdr_2022.csv: 0 duplicated unique IDs
cdr_2023.csv: 0 duplicated unique IDs
cdr_2024.csv: 0 duplicated unique IDs
PASS: No duplicated unique IDs were found.


True

## Test 10 — Display dataset summary

In [63]:
summaries = []
for year, path in CDR_FILES.items():
    try:
        frame = read_cdr(path)
        summaries.append({
            "file": path.name,
            "expected_year": year,
            "rows": len(frame),
            "columns": frame.shape[1],
            "size_mb": round(path.stat().st_size / (1024 * 1024), 2),
            "missing_cells": int(frame.isna().sum().sum()),
            "duplicate_rows": int(frame.duplicated().sum()),
        })
    except Exception as error:
        summaries.append({"file": path.name, "error": str(error)})
summary_table = pd.DataFrame(summaries)
display(summary_table)
show_result(not summary_table.empty, "Dataset summary created.", "No summary was created.")

,file,expected_year,rows,columns,size_mb,missing_cells,duplicate_rows
0,cdr_2021.csv,2021,151825,7,11.61,2087,0
1,cdr_2022.csv,2022,206623,7,15.83,2842,0
2,cdr_2023.csv,2023,212092,7,16.25,2897,0
3,cdr_2024.csv,2024,240375,7,18.43,3256,0


PASS: Dataset summary created.


True

In [64]:
from IPython.display import Markdown, display
from datetime import datetime

ALLOWED_DISPOSITIONS = {
    "answered",
    "abandoned",
    "no answer",
    "busy",
    "voicemail",
    "failed",
}

test_results = []
dataset_summaries = []

for expected_year, path in CDR_FILES.items():

    # Test 1: File exists
    exists = path.exists()

    # Test 2: File is not empty
    not_empty = exists and path.stat().st_size > 0

    # Test 3: File is under 25 MB
    under_25mb = (
        exists
        and path.stat().st_size <= 25 * 1024 * 1024
    )

    read_success = False
    seven_columns = False
    all_dates_valid = False
    correct_year = False
    dispositions_valid = False
    unique_ids_valid = False

    rows = 0
    columns = 0
    size_mb = 0
    missing_cells = 0
    duplicate_rows = 0
    invalid_dates = 0
    duplicate_ids = 0
    years_found = []
    unexpected_dispositions = []
    error_message = ""

    if exists:
        size_mb = round(
            path.stat().st_size / (1024 * 1024),
            2,
        )

        try:
            frame = read_cdr(path)
            read_success = True

            rows = len(frame)
            columns = frame.shape[1]
            missing_cells = int(
                frame.isna().sum().sum()
            )
            duplicate_rows = int(
                frame.duplicated().sum()
            )

            # Test 5: Seven columns
            seven_columns = columns == 7

            if seven_columns:

                # Test 6: Date validity
                parsed_dates = pd.to_datetime(
                    frame.iloc[:, 2],
                    format="%Y-%b-%d %I:%M:%S %p",
                    errors="coerce",
                )

                invalid_dates = int(
                    parsed_dates.isna().sum()
                )

                valid_date_percentage = (
                (len(frame) - invalid_dates) / len(frame)) * 100
                all_dates_valid = valid_date_percentage >= 99.0

                # Test 7: Expected year
                years_found = sorted(
                    parsed_dates
                    .dropna()
                    .dt.year
                    .unique()
                    .tolist()
                )

                correct_year = years_found == [
                    expected_year
                ]

                # Test 8: Dispositions
                dispositions = set(
                    frame.iloc[:, 4]
                    .dropna()
                    .str.strip()
                    .str.casefold()
                    .unique()
                )

                unexpected_dispositions = sorted(
                    dispositions
                    - ALLOWED_DISPOSITIONS
                )

                dispositions_valid = (
                    len(unexpected_dispositions) == 0
                )

                # Test 9: Unique IDs
                duplicate_ids = int(
                    frame.iloc[:, 5]
                    .dropna()
                    .duplicated()
                    .sum()
                )

                unique_ids_valid = duplicate_ids == 0

        except Exception as error:
            error_message = str(error)

    dataset_summaries.append({
        "file": path.name,
        "expected_year": expected_year,
        "rows": rows,
        "columns": columns,
        "size_mb": size_mb,
        "missing_cells": missing_cells,
        "duplicate_rows": duplicate_rows,
        "invalid_dates": invalid_dates,
        "duplicate_ids": duplicate_ids,
    })

    individual_results = [
        ("Test 1", "File exists", exists),
        ("Test 2", "File is not empty", not_empty),
        ("Test 3", "File is under 25 MB", under_25mb),
        ("Test 4", "CSV can be read", read_success),
        ("Test 5", "Dataset has seven columns", seven_columns),
        ("Test 6", "At least 99%  of dates are valid", all_dates_valid),
        ("Test 7", "Dataset contains expected year", correct_year),
        (
            "Test 8",
            "Disposition values are recognized",
            dispositions_valid,
        ),
        (
            "Test 9",
            "Unique IDs are not duplicated",
            unique_ids_valid,
        ),
    ]

    for test_id, description, passed in individual_results:
        test_results.append({
            "file": path.name,
            "test": test_id,
            "description": description,
            "status": "PASS" if passed else "FAIL",
        })


# Test 10: Summary creation
summary_table = pd.DataFrame(dataset_summaries)

summary_created = not summary_table.empty

test_results.append({
    "file": "All datasets",
    "test": "Test 10",
    "description": "Dataset summary created",
    "status": "PASS" if summary_created else "FAIL",
})


# Count final results
results_table = pd.DataFrame(test_results)

passed_count = int(
    (results_table["status"] == "PASS").sum()
)

failed_count = int(
    (results_table["status"] == "FAIL").sum()
)

total_count = len(results_table)


# Convert dataset summary to Markdown
summary_markdown = summary_table.to_markdown(
    index=False
)

results_markdown = results_table.to_markdown(
    index=False
)


# Create final Markdown report
report = f"""
# Erlang C Dataset Testing Report

## Test information

- Test date: {datetime.now().strftime("%Y-%m-%d %H:%M")}
- Number of datasets: {len(CDR_FILES)}
- Expected years: 2021, 2022, 2023 and 2024
- Maximum file size: 25 MB

## Overall results

| Result | Count |
|---|---:|
| Total checks | {total_count} |
| Passed | {passed_count} |
| Failed | {failed_count} |

## Dataset summary

{summary_markdown}

## Detailed test results

{results_markdown}

## Test descriptions

| Test | Check |
|---|---|
| Test 1 | All four dataset files exist |
| Test 2 | Dataset files are not empty |
| Test 3 | Dataset files are under 25 MB |
| Test 4 | CSV files can be read |
| Test 5 | Every dataset has seven columns |
| Test 6 | At least 99% of dates are valid and invalid dates can be removed |
| Test 7 | Every dataset contains the expected year |
| Test 8 | Disposition values are recognized |
| Test 9 | Unique IDs are not duplicated |
| Test 10 | Dataset summary is generated |

## Final status

{"PASS — All checks passed." if failed_count == 0 else f"FAIL — {failed_count} check(s) failed and should be reviewed."}

## Important note

A failed data-quality check does not mean that the testing
notebook is broken. It means that the test found a condition
in the dataset that does not satisfy the expected rule.
"""

display(Markdown(report))


# Erlang C Dataset Testing Report

## Test information

- Test date: 2026-09-04 21:06
- Number of datasets: 4
- Expected years: 2021, 2022, 2023 and 2024
- Maximum file size: 25 MB

## Overall results

| Result | Count |
|---|---:|
| Total checks | 37 |
| Passed | 37 |
| Failed | 0 |

## Dataset summary

| file         |   expected_year |   rows |   columns |   size_mb |   missing_cells |   duplicate_rows |   invalid_dates |   duplicate_ids |
|:-------------|----------------:|-------:|----------:|----------:|----------------:|-----------------:|----------------:|----------------:|
| cdr_2021.csv |            2021 | 151825 |         7 |     11.61 |            2087 |                0 |             168 |               0 |
| cdr_2022.csv |            2022 | 206623 |         7 |     15.83 |            2842 |                0 |             210 |               0 |
| cdr_2023.csv |            2023 | 212092 |         7 |     16.25 |            2897 |                0 |             218 |               0 |
| cdr_2024.csv |            2024 | 240375 |         7 |     18.43 |            3256 |                0 |             230 |               0 |

## Detailed test results

| file         | test    | description                       | status   |
|:-------------|:--------|:----------------------------------|:---------|
| cdr_2021.csv | Test 1  | File exists                       | PASS     |
| cdr_2021.csv | Test 2  | File is not empty                 | PASS     |
| cdr_2021.csv | Test 3  | File is under 25 MB               | PASS     |
| cdr_2021.csv | Test 4  | CSV can be read                   | PASS     |
| cdr_2021.csv | Test 5  | Dataset has seven columns         | PASS     |
| cdr_2021.csv | Test 6  | At least 99%  of dates are valid  | PASS     |
| cdr_2021.csv | Test 7  | Dataset contains expected year    | PASS     |
| cdr_2021.csv | Test 8  | Disposition values are recognized | PASS     |
| cdr_2021.csv | Test 9  | Unique IDs are not duplicated     | PASS     |
| cdr_2022.csv | Test 1  | File exists                       | PASS     |
| cdr_2022.csv | Test 2  | File is not empty                 | PASS     |
| cdr_2022.csv | Test 3  | File is under 25 MB               | PASS     |
| cdr_2022.csv | Test 4  | CSV can be read                   | PASS     |
| cdr_2022.csv | Test 5  | Dataset has seven columns         | PASS     |
| cdr_2022.csv | Test 6  | At least 99%  of dates are valid  | PASS     |
| cdr_2022.csv | Test 7  | Dataset contains expected year    | PASS     |
| cdr_2022.csv | Test 8  | Disposition values are recognized | PASS     |
| cdr_2022.csv | Test 9  | Unique IDs are not duplicated     | PASS     |
| cdr_2023.csv | Test 1  | File exists                       | PASS     |
| cdr_2023.csv | Test 2  | File is not empty                 | PASS     |
| cdr_2023.csv | Test 3  | File is under 25 MB               | PASS     |
| cdr_2023.csv | Test 4  | CSV can be read                   | PASS     |
| cdr_2023.csv | Test 5  | Dataset has seven columns         | PASS     |
| cdr_2023.csv | Test 6  | At least 99%  of dates are valid  | PASS     |
| cdr_2023.csv | Test 7  | Dataset contains expected year    | PASS     |
| cdr_2023.csv | Test 8  | Disposition values are recognized | PASS     |
| cdr_2023.csv | Test 9  | Unique IDs are not duplicated     | PASS     |
| cdr_2024.csv | Test 1  | File exists                       | PASS     |
| cdr_2024.csv | Test 2  | File is not empty                 | PASS     |
| cdr_2024.csv | Test 3  | File is under 25 MB               | PASS     |
| cdr_2024.csv | Test 4  | CSV can be read                   | PASS     |
| cdr_2024.csv | Test 5  | Dataset has seven columns         | PASS     |
| cdr_2024.csv | Test 6  | At least 99%  of dates are valid  | PASS     |
| cdr_2024.csv | Test 7  | Dataset contains expected year    | PASS     |
| cdr_2024.csv | Test 8  | Disposition values are recognized | PASS     |
| cdr_2024.csv | Test 9  | Unique IDs are not duplicated     | PASS     |
| All datasets | Test 10 | Dataset summary created           | PASS     |

## Test descriptions

| Test | Check |
|---|---|
| Test 1 | All four dataset files exist |
| Test 2 | Dataset files are not empty |
| Test 3 | Dataset files are under 25 MB |
| Test 4 | CSV files can be read |
| Test 5 | Every dataset has seven columns |
| Test 6 | At least 99% of dates are valid and invalid dates can be removed |
| Test 7 | Every dataset contains the expected year |
| Test 8 | Disposition values are recognized |
| Test 9 | Unique IDs are not duplicated |
| Test 10 | Dataset summary is generated |

## Final status

PASS — All checks passed.

## Important note

A failed data-quality check does not mean that the testing
notebook is broken. It means that the test found a condition
in the dataset that does not satisfy the expected rule.
